# Text-only pre-probe — ARCHITECTURE §7.7 item 1

> `system 블록 + 닫히지 않는 assistant + PAD 섞인 스트림`에서 Qwen의 지시 준수·응답 품질이
> 유지되는가 (non-thinking 모드 기준)

TRAINING_CURRICULUM §0 lists this as a Phase-0 deliverable, and ARCHITECTURE §7.7 marks it
**"오디오 파이프라인 없이 텍스트만으로 가능"** — no Mimi, no Moshi, no Haan code. Just Qwen3 and a
prompt shape. That is why it is a standalone notebook and imports nothing from the repo: there is
no vendored source here, so nothing can drift out of sync with it.

### What is being decided

ARCHITECTURE §7.3 removes turn markers from the dialogue region and lets PAD absorb their job:

> - **대화 구간에 `<|im_start|>`/`<|im_end|>` 턴 마커를 넣지 않는다.**
> - **`<|im_end|>`는 세션 종료에만** 사용한다.
> - Qwen이 발화 완결 시 `<|im_end|>`를 내려는 성향은, 파인튜닝에서 **"발화 완결 = PAD로 전환"으로
>   재매핑**한다.

If instruction-following collapses under that prompt shape, §7.3 and the Zone A/B/C layout (§7.2)
need rethinking — and that is a far more expensive discovery after training starts than before.

### What this notebook measures

| | question | why it is answerable today |
|---|---|---|
| **A. Instruction adherence** | does the system block still steer the answer? | pure inference |
| **B. `<|im_end|>` pull** | how strongly does Qwen want to close the turn? | this IS the re-mapping cost §7.3 defers -- **the one result here with no PAD confound** |
| **C. Degeneration** | does the output stay coherent? | repetition rate needs no reference text |

### One honesty caveat, stated up front

PAD (`151669`) and EPAD (`151670`) are **reserved slots with untrained embedding rows**
(`configs/tokens.yaml`: Qwen3-8B has 151936 rows but only 151669 real tokens). Feeding them at
probe time hands the model an *uninitialised* vector, so any degradation here is an **upper bound**
on the real post-finetune damage, not an estimate of it. The structural conditions (open block, no
turn markers) carry no such caveat — those are exact.

## 0. Setup

In [ ]:
# --- transformers: the project's FORK, not stock ---
#
# pyproject.toml:33 pins the dependency to latentforge/transformers-moshi, and uv.lock resolves it
# to the revision below. The probe in this notebook is Qwen3-only and would run on stock
# transformers, but it is a probe whose result decides an architecture question -- so it should run
# on the exact library the project trains with, not a near-neighbour. Installing the fork also
# makes Mimi/Moshi importable here, which is what the round-trip TODO at the bottom would need.
import importlib.util, os, subprocess, sys

FORK_REV = "5248a2e49cc02d2784d80757e015e60d907f9d6c"   # uv.lock
FORK = f"transformers @ git+https://github.com/latentforge/transformers-moshi.git@{FORK_REV}"


def _transformers_dir():
    # find_spec on a TOP-LEVEL name locates the package without executing it, so this is safe to
    # call before and after a pip install without leaving a stale module imported.
    spec = importlib.util.find_spec("transformers")
    if spec is None or not spec.submodule_search_locations:
        return None
    return list(spec.submodule_search_locations)[0]


def _has_fork():
    # `generation_moshi.py` exists ONLY in the fork -- upstream transformers ships Moshi but keeps
    # its generation loop inside modeling_moshi.py. Probing for `modeling_moshi` instead would pass
    # on stock transformers and silently skip this install.
    d = _transformers_dir()
    return bool(d) and os.path.isfile(os.path.join(d, "models", "moshi", "generation_moshi.py"))


if _has_fork():
    print("transformers fork already installed")
else:
    already_imported = "transformers" in sys.modules
    print("installing the transformers fork (a few minutes)...")
    # NOTE: `torch` is deliberately NOT in this list. Colab ships a matched torch/torchvision
    # pair, and torchvision's C++ ops are compiled against that exact torch -- upgrading torch
    # leaves torchvision importable but broken, and the failure surfaces far from its cause as
    # "RuntimeError: operator torchvision::nms does not exist", which transformers then reports
    # as "Could not import module 'Qwen3ForCausalLM'".
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--upgrade",
                    FORK, "accelerate", "safetensors", "huggingface_hub"], check=True)
    importlib.invalidate_caches()
    if not _has_fork():
        raise RuntimeError("the fork installed but generation_moshi.py is still missing")
    if already_imported:
        raise RuntimeError(
            "Stock transformers was already imported in this kernel, so the freshly installed fork "
            "will not be picked up.\n\nRuntime -> Restart session, then run this notebook again. "
            "The install is cached, so the second run skips it.")
    print("transformers fork installed")

# transformers imports torchvision lazily for vision models. A torchvision whose compiled ops no
# longer match the installed torch is WORSE than an absent one: `is_torchvision_available()` returns
# True, the import is attempted, and it dies. Nothing in this text-only probe needs it (ARCH 7.7:
# "오디오 파이프라인 없이 텍스트만으로 가능"), so a broken one is simply removed.
def _torchvision_broken():
    try:
        import torch, torchvision  # noqa: F401
        torch.ops.torchvision.nms   # touches the C++ registration that breaks on a torch mismatch
        return False
    except ImportError:
        return False                # absent is fine -- transformers skips it
    except Exception:
        return True                 # present but unusable


if _torchvision_broken():
    print("torchvision is present but its compiled ops do not match torch; removing it "
          "(unused by this probe)")
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchvision"], check=False)
    importlib.invalidate_caches()
    for _stale in [m for m in list(sys.modules) if m.startswith(("torchvision", "transformers"))]:
        del sys.modules[_stale]

import torch, transformers
from transformers.models.moshi.generation_moshi import MoshiGenerationMixin  # fork-only import
print("torch       ", torch.__version__)
print("transformers", transformers.__version__, "(fork verified)")

try:
    import glob, json
    meta = glob.glob(os.path.join(os.path.dirname(_transformers_dir()),
                                  "transformers-*.dist-info", "direct_url.json"))
    rev = json.load(open(meta[0]))["vcs_info"]["commit_id"] if meta else None
    print("fork revision", rev, "(expected", FORK_REV + ")")
    if rev and rev != FORK_REV:
        print("  !! not the pinned revision -- this is not the library the project trains with")
except Exception as exc:
    print("could not read the installed revision:", exc)

# Qwen3-8B in bf16 is ~16 GB: needs a GPU runtime with >= 20 GB (A100 / L4 fine, a 16 GB T4 is not).
print()
print("cuda        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
if torch.cuda.is_available():
    print("vram        ", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

installing the transformers fork (a few minutes)...
transformers fork installed
torch        2.11.0+cu128
transformers 5.15.0.dev0 (fork verified)
fork revision 5248a2e49cc02d2784d80757e015e60d907f9d6c (expected 5248a2e49cc02d2784d80757e015e60d907f9d6c)

cuda         NVIDIA A100-SXM4-80GB
vram         79 GB


In [ ]:
# The backbone under test. ARCHITECTURE §1 makes Qwen3-8B the Temporal backbone, so 8B is the
# answer that counts; the smaller ids are for a fast structural smoke run only -- the *shape* of
# the effect transfers, the magnitude does not.
MODEL_ID = "Qwen/Qwen3-8B"      # or "Qwen/Qwen3-1.7B" / "Qwen/Qwen3-0.6B" for a quick check

# configs/tokens.yaml -- copied as literals rather than imported, so this notebook stays
# repo-independent. Qwen3-8B has 151936 embedding rows but only 151669 real tokens (highest id
# 151668), leaving 267 genuinely reserved slots.
TEXT_PAD_ID  = 151669   # stream PAD  -- "not speaking, session continues" (~65% of text tokens)
TEXT_EPAD_ID = 151670   # EPAD        -- inserted one frame before a word starts (§5.0.1)
REAL_VOCAB   = 151669   # ids >= this are untrained reserved rows

# ARCHITECTURE §7.5 / RISKS §7.11: live operation is non-thinking; thinking would blow the
# 12.5 tok/s text-channel budget. Every generation below runs with it off.
ENABLE_THINKING = False

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto"
).eval()

print("vocab (tokenizer)   :", len(tokenizer))
print("embedding rows      :", model.get_input_embeddings().weight.shape[0])
spare = model.get_input_embeddings().weight.shape[0] - len(tokenizer)
print("reserved spare rows :", spare)
assert TEXT_PAD_ID >= len(tokenizer), (
    f"PAD id {TEXT_PAD_ID} is an EXISTING token ({tokenizer.convert_ids_to_tokens(TEXT_PAD_ID)!r}); "
    "the whole point is that it occupies an untrained slot (ARCHITECTURE §7.6)")
assert spare > 0, "no reserved rows -- the token-slot plan in configs/tokens.yaml does not apply here"
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
print("im_end id           :", IM_END_ID)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

vocab (tokenizer)   : 151669
embedding rows      : 151936
reserved spare rows : 267
im_end id           : 151645


## 1. Building the prompt shapes

Four conditions, each isolating one step away from stock ChatML. Only by holding the others fixed
can a drop be attributed to the thing that caused it.

| condition | turn markers | assistant block | PAD |
|---|---|---|---|
| `chatml` | yes | closed properly | none |
| `open` | yes | **never closed** | none |
| `open_pad` | yes | never closed | **interleaved** |
| `no_markers` | **none in dialogue** | never closed | interleaved |

`no_markers` is the shape ARCHITECTURE §7.2/§7.3 actually specifies. The three before it exist so a
failure can be pinned on the open block, the PAD, or the marker removal — separately.

In [ ]:
import random

def interleave_pad(token_ids, pad_ratio, pad_id=TEXT_PAD_ID, epad_id=TEXT_EPAD_ID, seed=0):
    '''Spread PAD through a token stream at the given ratio, with EPAD one slot before each word.

    ARCHITECTURE §5.0.1: PAD is the between-word slot (~65% of English dialogue text tokens) and
    EPAD is inserted one frame BEFORE a word starts -- "padding is ending, a word begins next".
    The EPAD placement is what makes this a faithful stream rather than random noise injection.
    '''
    rng = random.Random(seed)
    out = []
    for tok in token_ids:
        # PAD run before this word, then exactly one EPAD as the onset trigger.
        n_pad = 0
        while rng.random() < pad_ratio:
            n_pad += 1
        if n_pad:
            out.extend([pad_id] * (n_pad - 1) + [epad_id])
        out.append(tok)
    return out


def _ids(text):
    '''Plain list[int] for a raw string -- no special tokens, they are placed explicitly below.'''
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def _chat_text(messages, add_generation_prompt):
    '''Render a ChatML string.

    `tokenize=False` on purpose: with `tokenize=True` this returns a `BatchEncoding`, not a
    `list[int]`, and concatenating that with a list silently produces something `torch.tensor`
    later rejects with "Could not infer dtype of tokenizers.Encoding" -- a failure far from its
    cause. Rendering to text and tokenizing separately keeps every branch below on plain lists.
    '''
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
        enable_thinking=ENABLE_THINKING,
    )


def build_prompt(system, dialogue, condition, pad_ratio=0.65, seed=0):
    '''Return input_ids for one condition. Zone A (system) is always a proper ChatML block.'''
    sys_block = _ids(_chat_text([{"role": "system", "content": system}], False))
    body = _ids(dialogue)

    if condition == "chatml":
        return _ids(_chat_text(
            [{"role": "system", "content": system}, {"role": "user", "content": dialogue}], True))
    if condition == "open":
        head = _ids("<|im_start|>assistant\n")
        return sys_block + head + body
    if condition == "open_pad":
        head = _ids("<|im_start|>assistant\n")
        return sys_block + head + interleave_pad(body, pad_ratio, seed=seed)
    if condition == "no_markers":
        # ARCHITECTURE §7.3: no turn markers inside the dialogue region at all.
        return sys_block + interleave_pad(body, pad_ratio, seed=seed)
    raise ValueError(condition)


# Every branch must yield a plain list[int]; anything else fails much later inside torch.tensor.
for _cond in ("chatml", "open", "open_pad", "no_markers"):
    _ids_out = build_prompt("You are helpful.", "Hello there friend.", _cond, seed=0)
    assert isinstance(_ids_out, list) and all(isinstance(t, int) for t in _ids_out), (
        f"build_prompt({_cond!r}) returned {type(_ids_out).__name__}, not list[int]")
print("all four conditions return list[int]")

# Sanity: PAD really lands at the requested density, and every EPAD precedes a real token.
_probe = tokenizer("hello world foo bar baz", add_special_tokens=False)["input_ids"]
_mixed = interleave_pad(_probe, 0.65, seed=1)
_pads = sum(t in (TEXT_PAD_ID, TEXT_EPAD_ID) for t in _mixed)
print(f"{len(_probe)} word tokens -> {len(_mixed)} stream tokens, {_pads} pad/epad "
      f"({_pads / len(_mixed):.0%} of the stream)")
for i, t in enumerate(_mixed[:-1]):
    if t == TEXT_EPAD_ID:
        assert _mixed[i + 1] not in (TEXT_PAD_ID, TEXT_EPAD_ID), "EPAD must sit one slot before a word"
print("EPAD placement verified")

all four conditions return list[int]
5 word tokens -> 11 stream tokens, 6 pad/epad (55% of the stream)
EPAD placement verified


## 2. Measurement A — does the system block still steer the answer?

The system instruction asks for something machine-checkable, so adherence is a number rather than an
impression. Each probe pairs an instruction with a predicate over the generated text.

**Read `open` as the headline, not `no_markers`.** `open` (turn markers, unclosed block, NO PAD) is
the only condition that changes the *structure* without also feeding untrained PAD/EPAD embeddings,
so it is the one whose adherence score means what it says. `open_pad` and `no_markers` add PAD, whose
embedding rows are random init here -- a collapse there is an **upper bound on damage**, not a
measurement of it (see §0). What those two DO measure honestly is the **empty-output rate**: how
often the untrained PAD makes the model close the turn immediately. That is reported separately below
and must never be read as compliance -- an empty answer trivially satisfies "at most 10 words".

In [ ]:
import re

def korean_ratio(text):
    hangul = len(re.findall(r"[가-힣]", text))
    letters = len(re.findall(r"[가-힣A-Za-z]", text))
    return hangul / letters if letters else 0.0

def english_ratio(text):
    return 1.0 - korean_ratio(text) if re.search(r"[가-힣A-Za-z]", text) else 0.0

PROBES = [
    {"name": "answer-in-korean",
     "system": "You are a helpful assistant. Always answer in Korean, no matter what language the user writes in.",
     "dialogue": "Explain in two sentences why the sky appears blue.",
     "check": korean_ratio, "target": "Korean character ratio"},
    {"name": "answer-in-english",
     "system": "You are a helpful assistant. Always answer in English only.",
     "dialogue": "하늘이 파랗게 보이는 이유를 두 문장으로 설명해줘.",
     "check": english_ratio, "target": "English character ratio"},
    {"name": "stay-terse",
     "system": "You are terse. Answer in at most 10 words. Never exceed that.",
     "dialogue": "What is the capital of France, and why is it famous?",
     "check": lambda t: 1.0 if len(t.split()) <= 10 else 10.0 / max(len(t.split()), 1),
     "target": "<=10 word compliance"},
]

@torch.no_grad()
def generate(input_ids, max_new_tokens=80, seed=0):
    '''Generate, and report WHY the output is what it is.

    Returns (text, info). `info["first_is_special"]` is the tell for this probe: an empty `text`
    almost always means the model emitted <|im_end|> (or another special id) as its very first
    step -- the untrained PAD embedding gave it no signal, so it fell back to its strongest habit
    (measurement B: P(im_end) ~ 0.99 at completion) and closed the turn immediately. Without this
    field an empty string is indistinguishable from a real refusal.
    '''
    torch.manual_seed(seed)
    ids = torch.tensor([input_ids], device=model.device)
    gen = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    new_ids = gen[0, ids.shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    first = int(new_ids[0]) if new_ids.numel() else None
    info = {
        "first_is_special": first in tokenizer.all_special_ids if first is not None else True,
        "n_new_tokens": int(new_ids.numel()),
    }
    return text, info

In [ ]:
import statistics

CONDITIONS = ["chatml", "open", "open_pad", "no_markers"]
N_SAMPLES = 3   # raise for a tighter estimate; sampling noise is real at 3

def _is_empty(text):
    return len(text.strip()) == 0


results = {}
for probe in PROBES:
    for cond in CONDITIONS:
        scores, empties, texts = [], [], []
        for seed in range(N_SAMPLES):
            ids = build_prompt(probe["system"], probe["dialogue"], cond, seed=seed)
            text, info = generate(ids, seed=seed)
            texts.append(text)
            if _is_empty(text):
                empties.append(1.0)
            else:
                empties.append(0.0)
                scores.append(probe["check"](text))   # compliance is scored ONLY on real output
        results[(probe["name"], cond)] = {
            # mean over NON-EMPTY samples; None when everything came back empty (nothing to score).
            "score": statistics.mean(scores) if scores else None,
            "empty_rate": statistics.mean(empties),
            "texts": texts,
        }

print(f"{'probe':20s} {'condition':12s} {'score':>7s} {'empty':>7s}   (score = compliance among non-empty)")
print("-" * 66)
for probe in PROBES:
    base = results[(probe["name"], "open")]["score"]   # `open`, not chatml: the no-PAD structural baseline
    for cond in CONDITIONS:
        r = results[(probe["name"], cond)]
        score_str = "  -  " if r["score"] is None else f"{r['score']:5.2f}"
        delta = ""
        if cond != "open" and r["score"] is not None and base is not None:
            delta = f"  {r['score'] - base:+.2f} vs open"
        flag = "  <- all empty" if r["empty_rate"] == 1.0 else ""
        print(f"{probe['name']:20s} {cond:12s} {score_str:>7s} {r['empty_rate']:6.0%}{delta}{flag}")
    print()
print("An 'all empty' row is the untrained-PAD collapse, NOT a compliance result. The number that")
print("carries meaning is the `open` column and the empty_rate itself.")

probe                condition      score   empty   (score = compliance among non-empty)
------------------------------------------------------------------
answer-in-korean     chatml          1.00     0%  +0.87 vs open
answer-in-korean     open            0.13     0%
answer-in-korean     open_pad         -     100%  <- all empty
answer-in-korean     no_markers       -     100%  <- all empty

answer-in-english    chatml          0.00     0%
answer-in-english    open             -     100%  <- all empty
answer-in-english    open_pad         -     100%  <- all empty
answer-in-english    no_markers       -     100%  <- all empty

stay-terse           chatml          1.00     0%
stay-terse           open             -     100%  <- all empty
stay-terse           open_pad         -     100%  <- all empty
stay-terse           no_markers       -     100%  <- all empty

An 'all empty' row is the untrained-PAD collapse, NOT a compliance result. The number that
carries meaning is the `open` colum

In [ ]:
# Read the actual generations, not only the scores -- a metric can look fine while the text is
# visibly broken, and this probe exists to be judged by a human (TRAINING_CURRICULUM §0).
for probe in PROBES[:1]:
    for cond in CONDITIONS:
        text = results[(probe["name"], cond)]["texts"][0]
        shown = text[:400] if text.strip() else "(empty -- model closed the turn immediately; see measurement B)"
        print(f"--- {probe['name']} / {cond} " + "-" * 30)
        print(shown)
        print()

--- answer-in-korean / chatml ------------------------------
하늘은 파란색으로 보이는 이유는 태양광이 대기 중의 분자에 의해 산란되기 때문입니다. 특히 파란 빛은 짧은 파장으로 인해 더 많이 산란되어, 우리 눈에 더 많이 닿기 때문에 하늘이 파랗게 보입니다.

--- answer-in-korean / open ------------------------------
 The sky appears blue because sunlight is scattered by the atmosphere, and shorter wavelengths like blue light are scattered more than longer wavelengths. This scattering effect, known as Rayleigh scattering, makes the sky appear blue to our eyes.
</think>

하늘이 파란 이유는 태양빛이 대기 중에서 산란되는 현상 때문입니다. 짧은 파장의 빛

--- answer-in-korean / open_pad ------------------------------
(empty -- model closed the turn immediately; see measurement B)

--- answer-in-korean / no_markers ------------------------------
(empty -- model closed the turn immediately; see measurement B)



## 3. Measurement B — how hard does Qwen pull toward `<|im_end|>`?  (the load-bearing result)

ARCHITECTURE §7.3 defers this to finetuning:

> Qwen이 발화 완결 시 `<|im_end|>`를 내려는 성향은, 파인튜닝에서 **"발화 완결 = PAD로 전환"으로
> 재매핑**한다. 구조를 새로 발명하는 것보다 국소적인 행동 재매핑이라 비용이 작다.

That claim has a measurable cost, and unlike measurement A this one is **not confounded by the
untrained PAD** -- it is a single forward pass over a finished utterance, no generation, no PAD fed.
`P(<|im_end|>)` at the completion point is exactly the habit the finetune must move. A small
probability means a light touch; a dominant one means the finetune is fighting the backbone's
strongest reflex at ~65% of all text positions (every PAD slot).

Measured over a spread of completed utterances -- English / Korean, statement / question / list --
so a single lucky or unlucky sentence cannot set the conclusion.

In [ ]:
@torch.no_grad()
def next_token_probs(input_ids):
    ids = torch.tensor([input_ids], device=model.device)
    logits = model(ids).logits[0, -1].float()
    return torch.softmax(logits, dim=-1)

SYS = "You are a helpful assistant. Answer briefly."
COMPLETED = [
    (SYS, "The capital of France is Paris."),
    (SYS, "네, 도와드리겠습니다."),
    (SYS, "That's an interesting question."),
    (SYS, "Water boils at 100 degrees Celsius at sea level."),
    (SYS, "감사합니다. 좋은 하루 되세요."),
    (SYS, "The meeting is scheduled for three o'clock."),
    (SYS, "그 영화는 정말 재미있었어요."),
    (SYS, "Photosynthesis converts sunlight into chemical energy."),
    (SYS, "저는 서울에 살고 있습니다."),
    (SYS, "The answer is forty-two."),
    (SYS, "What time does the store open tomorrow?"),
    (SYS, "이 문제는 어떻게 풀어야 하나요?"),
    (SYS, "My favorite colors are blue, green, and red."),
    (SYS, "일정은 월요일, 수요일, 금요일입니다."),
    (SYS, "Thank you for your help today."),
]

print(f"{'utterance':44s} {'P(im_end)':>10s} {'rank':>6s}")
print("-" * 62)
rows = []
for system, utterance in COMPLETED:
    # Plain ChatML, NO PAD: this measures the backbone's native completion reflex, which is what
    # the finetune inherits. (Feeding untrained PAD here would only add noise, not signal.)
    ids = build_prompt(system, utterance, "open", seed=0)
    probs = next_token_probs(ids)
    p_end = float(probs[IM_END_ID])
    rank = int((probs > p_end).sum()) + 1
    rows.append((p_end, rank))
    print(f"{utterance[:42]:44s} {p_end:10.4f} {rank:6d}")

import statistics as _st
p_ends = [r[0] for r in rows]
ranks = [r[1] for r in rows]
print("-" * 62)
print(f"{'MEAN':44s} {_st.mean(p_ends):10.4f} {_st.mean(ranks):6.1f}")
print(f"{'MEDIAN':44s} {_st.median(p_ends):10.4f} {_st.median(ranks):6.0f}")
print(f"rank-1 (im_end is the single most likely next token): "
      f"{sum(r == 1 for r in ranks)}/{len(ranks)} utterances")
print()
print("This is the number to carry into the Phase 1 design. A median rank of 1 means closing the")
print("turn is the backbone's DEFAULT at a completion point -- so the PAD-transition re-mapping is")
print("not the 'local, low-cost' touch ARCHITECTURE §7.3 assumed, and its convergence is worth")
print("monitoring as a first-class training signal, not taking for granted.")

utterance                                     P(im_end)   rank
--------------------------------------------------------------
The capital of France is Paris.                  0.1694      2
네, 도와드리겠습니다.                                     0.0000  50520
That's an interesting question.                  0.0003     54
Water boils at 100 degrees Celsius at sea        0.0265      6
감사합니다. 좋은 하루 되세요.                                0.3529      1
The meeting is scheduled for three o'clock       0.0126      7
그 영화는 정말 재미있었어요.                                 0.0009    109
Photosynthesis converts sunlight into chem       0.0374      4
저는 서울에 살고 있습니다.                                  0.0036     35
The answer is forty-two.                         0.1084      3
What time does the store open tomorrow?          0.5032      1
이 문제는 어떻게 풀어야 하나요?                               0.0686      2
My favorite colors are blue, green, and re       0.0002     24
일정은 월요일, 수요일, 금요일입니다.                            0.0015

In [ ]:
@torch.no_grad()
def next_token_probs(input_ids):
    ids = torch.tensor([input_ids], device=model.device)
    logits = model(ids).logits[0, -1].float()
    return torch.softmax(logits, dim=-1)

SYS = "You are a helpful assistant. Answer briefly."
COMPLETED = [
    (SYS, "The capital of France is Paris."),
    (SYS, "네, 도와드리겠습니다."),
    (SYS, "That's an interesting question."),
    (SYS, "Water boils at 100 degrees Celsius at sea level."),
    (SYS, "감사합니다. 좋은 하루 되세요."),
    (SYS, "The meeting is scheduled for three o'clock."),
    (SYS, "그 영화는 정말 재미있었어요."),
    (SYS, "Photosynthesis converts sunlight into chemical energy."),
    (SYS, "저는 서울에 살고 있습니다."),
    (SYS, "The answer is forty-two."),
    (SYS, "What time does the store open tomorrow?"),
    (SYS, "이 문제는 어떻게 풀어야 하나요?"),
    (SYS, "My favorite colors are blue, green, and red."),
    (SYS, "일정은 월요일, 수요일, 금요일입니다."),
    (SYS, "Thank you for your help today."),
]

print(f"{'utterance':44s} {'P(im_end)':>10s} {'rank':>6s}")
print("-" * 62)
rows = []
for system, utterance in COMPLETED:
    # Plain ChatML, NO PAD: this measures the backbone's native completion reflex, which is what
    # the finetune inherits. (Feeding untrained PAD here would only add noise, not signal.)
    ids = build_prompt(system, utterance, "open", seed=0)
    probs = next_token_probs(ids)
    p_end = float(probs[IM_END_ID])
    rank = int((probs > p_end).sum()) + 1
    rows.append((p_end, rank))
    print(f"{utterance[:42]:44s} {p_end:10.4f} {rank:6d}")

import statistics as _st
p_ends = [r[0] for r in rows]
ranks = [r[1] for r in rows]
print("-" * 62)
print(f"{'MEAN':44s} {_st.mean(p_ends):10.4f} {_st.mean(ranks):6.1f}")
print(f"{'MEDIAN':44s} {_st.median(p_ends):10.4f} {_st.median(ranks):6.0f}")
print(f"rank-1 (im_end is the single most likely next token): "
      f"{sum(r == 1 for r in ranks)}/{len(ranks)} utterances")
# Data-driven verdict, so the printed conclusion can never drift from the numbers above.
med_rank = _st.median(ranks)
med_p = _st.median(p_ends)
n_rank1 = sum(r == 1 for r in ranks)
print()
if med_rank <= 2:
    print(f"median rank {med_rank:.0f}: closing the turn is at or near the backbone's DEFAULT at a")
    print("completion point. ARCHITECTURE §7.3's '국소적·저비용 재매핑' is then OPTIMISTIC -- the")
    print("finetune fights the strongest reflex, and its convergence must be a monitored signal.")
else:
    print(f"median rank {med_rank:.0f} (P={med_p:.3f}), im_end at rank 1 for only {n_rank1}/{len(ranks)}:")
    print("closing the turn is NOT the backbone's default at a completion point. This is EVIDENCE FOR")
    print("ARCHITECTURE §7.3's 'low-cost re-mapping' assumption -- the reflex the finetune must move is")
    print("mild for most completions. Watch the outliers instead: the few high-P cases (typically")
    print("question-final) are where the re-mapping actually has work to do.")


utterance                                     P(im_end)   rank
--------------------------------------------------------------
The capital of France is Paris.                  0.1694      2
네, 도와드리겠습니다.                                     0.0000  50520
That's an interesting question.                  0.0003     54
Water boils at 100 degrees Celsius at sea        0.0265      6
감사합니다. 좋은 하루 되세요.                                0.3529      1
The meeting is scheduled for three o'clock       0.0126      7
그 영화는 정말 재미있었어요.                                 0.0009    109
Photosynthesis converts sunlight into chem       0.0374      4
저는 서울에 살고 있습니다.                                  0.0036     35
The answer is forty-two.                         0.1084      3
What time does the store open tomorrow?          0.5032      1
이 문제는 어떻게 풀어야 하나요?                               0.0686      2
My favorite colors are blue, green, and re       0.0002     24
일정은 월요일, 수요일, 금요일입니다.                            0.0015

## 4. Measurement C — does the output stay coherent?

A model can obey the system block and still degenerate into loops. Repetition needs no reference
text, so it is measurable here.

In [ ]:
def repetition_rate(text, n=3):
    '''Fraction of n-grams that are repeats. ~0 is healthy; >0.3 is looping.'''
    words = text.split()
    if len(words) < n + 1:
        return 0.0
    grams = [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    return 1.0 - len(set(grams)) / len(grams)

print(f"{'condition':12s} {'repetition':>11s} {'mean words':>11s}")
print("-" * 36)
for cond in CONDITIONS:
    texts = [t for probe in PROBES for t in results[(probe["name"], cond)]["texts"] if t.strip()]
    if not texts:
        print(f"{cond:12s} {'  -  ':>11s} {'  -  ':>11s}   (all empty -- nothing to score)")
        continue
    rep = statistics.mean(repetition_rate(t) for t in texts)
    length = statistics.mean(len(t.split()) for t in texts)
    print(f"{cond:12s} {rep:11.3f} {length:11.1f}")

condition     repetition  mean words
------------------------------------
chatml             0.000        19.9
open               0.000        51.0
open_pad             -           -     (all empty -- nothing to score)
no_markers           -           -     (all empty -- nothing to score)


## 5. PAD density sweep

ARCHITECTURE §7.6 puts stream PAD at **~65%** of English dialogue text tokens. This sweeps around
that figure to find where — if anywhere — adherence falls off, which tells you whether 65% sits on a
cliff or on a plateau.

In [ ]:
DENSITIES = [0.0, 0.3, 0.5, 0.65, 0.8]
probe = PROBES[0]

print(f"{'pad_ratio':>10s} {'stream len':>11s} {'empty':>7s} {'score':>7s}   (score = among non-empty)")
print("-" * 47)
for ratio in DENSITIES:
    lengths, empties, scores = [], [], []
    for seed in range(N_SAMPLES):
        ids = build_prompt(probe["system"], probe["dialogue"], "no_markers", pad_ratio=ratio, seed=seed)
        lengths.append(len(ids))
        text, _ = generate(ids, seed=seed)
        if text.strip():
            empties.append(0.0); scores.append(probe["check"](text))
        else:
            empties.append(1.0)
    score_str = "  -  " if not scores else f"{statistics.mean(scores):5.2f}"
    print(f"{ratio:10.2f} {statistics.mean(lengths):11.0f} {statistics.mean(empties):6.0%} {score_str:>7s}")
print()
print("At ratio 0.00 there is no PAD, so this row is the clean control. Any climb in `empty` as the")
print("ratio rises is the untrained-PAD collapse, not evidence about what a TRAINED PAD would do.")

 pad_ratio  stream len   empty   score   (score = among non-empty)
-----------------------------------------------
      0.00          36     0%    0.43
      0.30          39   100%     -  
      0.50          44   100%     -  
      0.65          57   100%     -  
      0.80          69   100%     -  

At ratio 0.00 there is no PAD, so this row is the clean control. Any climb in `empty` as the
ratio rises is the untrained-PAD collapse, not evidence about what a TRAINED PAD would do.


## 6. Verdict

Deliberately not auto-computed: TRAINING_CURRICULUM §0 asks for a human judgement, and no design doc
gives a threshold. Read the three measurements for what each can and cannot answer.

**What this probe CAN settle (no PAD confound):**
- **Unclosed assistant block is safe** if `open` adherence stays near `chatml`. This is the clean
  structural test -- a drop here would indict §7.3's "do not close the block" decision directly, and
  no PAD finetuning could fix it.
- **The `<|im_end|>` reflex (measurement B)** is the load-bearing number. A median rank of 1 means
  ARCHITECTURE §7.3's "국소적·저비용 재매핑" is optimistic: the finetune must overturn the backbone's
  default at every completion point. Carry this into Phase 1 as a monitored signal.

**What this probe CANNOT settle (PAD is untrained here):**
- Adherence under a PAD-mixed stream. `open_pad` / `no_markers` feed random-init PAD/EPAD rows, so a
  collapse there is an **upper bound on damage**, not a measurement. The honest readout from those
  two is the **empty_rate** -- how completely the untrained PAD derails generation -- never a
  compliance score. An empty answer trivially "complies" with a length limit, which is why empties
  are counted separately above and excluded from the score.

**The single most useful thing this run establishes:** the PAD-mixed question is *not answerable by
inference alone*. It requires a trained PAD embedding, which is §7.7 item 2 (a finetune), below.

---

### TODO — not in this notebook

- **ARCHITECTURE §7.7 item 2**: *"발화 완결 = PAD 전환" 재매핑이 얼마나 빨리 학습되는가* — needs an
  actual finetune, not inference. Measurement B here quantifies the starting point that finetune
  must move (the `<|im_end|>` reflex), and this run showed the PAD-adherence question cannot be
  reached any other way.
- **ARCHITECTURE §4.3 / TRAINING_CURRICULUM §0**: the Mimi Korean round-trip reconstruction test
  (encode → decode → ASR WER, plus minimal-pair checks for pitch accent and vowel length).
  Deferred: it needs a Korean ASR model, which is an open decision.